# 05 - Auditoria: modelo subrogado con reglas

**Objetivo:** ajustar un modelo subrogado interpretable (arbol de decision) que aproxime cada uno de
los dos modelos de caja negra, extraer reglas legibles y medir su fidelidad.

**Por que hace falta.** El modelo que decide es XGBoost: un ensemble de cientos de arboles cuya
salida ningun humano puede justificar leyendo sus parametros. El enunciado, sin embargo, exige poder
explicar la decision ("si un cliente pide explicaciones sobre por que se le deniega un credito, que
informacion le damos?"). El subrogado es la respuesta clasica: entrenamos un modelo *interpretable*
-un arbol poco profundo- cuyo objetivo NO es predecir el impago, sino **imitar a la caja negra**. Si
el arbol reproduce sus decisiones con fidelidad alta, sus reglas if/then son un retrato legible de
la logica que XGBoost aplica de hecho.

**La distincion critica (fidelidad != accuracy).** El subrogado se entrena contra
`modelo.predict(X)`, NUNCA contra `SeriousDlqin2yrs`. Y se evalua igual: la metrica es la
**fidelidad** (coincidencia con la caja negra), no la accuracy (acierto sobre la verdad). La razon es
conceptual, no tecnica: no estamos construyendo un modelo alternativo -ya tenemos uno-, sino un
EXPLICADOR. Un subrogado con fidelidad perfecta a un modelo mediocre seria un explicador perfecto de
un modelo mediocre, y eso es exactamente lo que queremos: describir lo que el modelo hace, aciertos y
errores incluidos. Ver `docs/teoria/subrogados.md`, seccion 3.4.

**Entregables que produce este notebook:**
- `results/tables/reglas_subrogado_coste1.md` y `reglas_subrogado_coste10.md`
- `results/figures/arbol_subrogado_coste1.png` y `arbol_subrogado_coste10.png`
- `results/tables/sub_05_fidelidad.csv` (tabla de fidelidad por escenario)
- `results/figures/sub_05_fidelidad_vs_profundidad.png` (curva del trade-off)

## Conectores

**Recibe:**
- `results/models/modelo_coste1.joblib` (de `03_modelo_coste1.ipynb`): bundle con el estimador
  XGBoost refiteado en train completo y su umbral simetrico (~0.4934).
- `results/models/modelo_coste10.joblib` (de `04_modelo_coste10.ipynb`): MISMO estimador XGBoost,
  umbral asimetrico (~0.0906). Recordatorio de D-0.2: un solo scoring, dos umbrales.
- `data/processed/test.parquet` (de `02_preprocesado.ipynb`): 21000 filas, 12 features. Es el
  conjunto sobre el que auditamos.

**Entrega** a `99_ENTREGA.ipynb` (seccion de auditoria):
- Reglas legibles por escenario, figuras de los arboles y la tabla de fidelidad.

**Por que auditamos sobre TEST y no sobre train.** El subrogado debe imitar a la caja negra en el
regimen donde esta opera, no donde memorizo. Sobre train, XGBoost esta sobreajustado y sus
decisiones son mas "limpias" de lo que seran en produccion; un subrogado ajustado ahi daria una
fidelidad optimista y unas reglas que no describen el comportamiento real. Test (21000 filas no
vistas) es la muestra honesta. No hay fuga posible: no estamos eligiendo hiperparametros del modelo
principal, solo describiendo un modelo YA CERRADO.

## Decisiones que afectan a este notebook

- **D-0.3** (familia de modelo): **CERRADA** a favor de **XGBoost**, un modelo de caja negra. Esto
  hace que el subrogado mantenga toda su relevancia: si el modelo principal fuera una regresion
  logistica, seria directamente interpretable y aproximarlo con reglas seria redundante. No es el
  caso.
- **D-0.2** (un scoring + dos umbrales): **CERRADA**. Los dos escenarios comparten el MISMO score de
  caja negra, pero su DECISION (`y_predicha`) difiere porque el umbral cambia (0.4934 vs 0.0906).
  Como el subrogado imita DECISIONES, no probabilidades, cada escenario necesita su propio arbol
  (secciones 2 y 3). Asumir que las reglas de uno valen para el otro seria un error.

- **D-5.1 (se cierra aqui): complejidad del arbol y metrica de fidelidad.**

  *Complejidad.* Se limita por **`max_depth`**, y no se fija a dedo: se barren las profundidades 2-6
  y se reporta la **curva fidelidad-vs-profundidad** (seccion 2). Asi el trade-off
  interpretabilidad<->fidelidad no se afirma, se DEMUESTRA, y la profundidad elegida queda
  justificada por el punto donde la curva se aplana (ganancia marginal de fidelidad que ya no
  compensa la perdida de legibilidad). La alternativa (`max_leaf_nodes`, patron del material de
  partida) es igual de valida; se prefiere `max_depth` porque produce reglas de longitud acotada
  ("si se cumplen estas <=D condiciones -> denegar"), que es justo el formato que un cliente o un
  regulador puede leer.

  *Metrica.* Se reportan **tres**, porque una sola engana:
  1. **Fidelidad global**: % de decisiones coincidentes con la caja negra. Es la cifra titular, pero
     es TRAMPOSA bajo desbalanceo (ver abajo).
  2. **Fidelidad por clase**: coincidencia dentro de los "concedidos" y dentro de los "denegados"
     por la caja negra, por separado.
  3. **F1 sobre la clase "denegar"**, tratando la salida de XGBoost como etiqueta.

  *Por que tres y no una.* En el escenario simetrico XGBoost solo deniega al **2.4%**. Un subrogado
  degenerado que dijera "conceder siempre" alcanzaria **~97.6% de fidelidad global** sin haber
  aprendido NADA y sin producir una sola regla util. Reportar solo la fidelidad global permitiria
  vender ese 97% como un exito; la fidelidad por clase y el F1 lo desenmascaran de inmediato. Este
  es el riesgo real del escenario simetrico y lo declaramos por adelantado.

## 1. Carga de los modelos entrenados y del conjunto de test

Cargamos los dos bundles de `03` y `04` y el `test.parquet`. De cada bundle extraemos el estimador
XGBoost y su umbral, y verificamos explicitamente la tesis de **D-0.2**: que el estimador es el mismo
objeto en ambos escenarios y que lo unico que cambia es el punto de corte.

Generamos aqui las dos **etiquetas objetivo** del subrogado, que son el corazon del notebook:
`y_bb_coste1` y `y_bb_coste10` = las decisiones de la caja negra en test bajo cada umbral. Insisto:
estas, y no `SeriousDlqin2yrs`, son lo que los arboles van a aprender a imitar.

In [1]:
# === Configuracion - notebook 05 (auditoria: subrogado) ===
# NOTA: este notebook NO importa src.modeling (que arrastra TensorFlow/Keras para los MLP).
# El subrogado solo necesita sklearn + xgboost + pandas. Cargamos los bundles con joblib
# directo: los .joblib de 03/04 contienen los estimadores YA entrenados, no hay que reentrenar.
import os
import sys
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import seaborn as sns
import joblib

from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import f1_score, precision_score, recall_score

# cwd = raiz del repo -> rutas SIN "../" (mismo contrato que 03/04).
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
import src.cost_utils as cu          # modulo PURO (numpy): no arrastra TF

RANDOM_STATE = 42
TARGET = cu.TARGET
np.random.seed(RANDOM_STATE)

# --- Paleta heredada de 01-04 (coherencia visual del proyecto) ---
COLOR_TARGET = {0: "#2a78d6", 1: "#e34948"}
COLOR_INK_PRIMARY = "#0b0b0b"
COLOR_INK_SECONDARY = "#52514e"
COLOR_MUTED = "#898781"
COLOR_GRID = "#e1e0d9"
COLOR_BASELINE = "#c3c2b7"
COLOR_GANADOR = "#1c5cab"
COLOR_ESC1 = "#8aa9cf"                # esc1 (simetrico)  - azul suave
COLOR_ESC2 = "#b5462f"                # esc2 (asimetrico) - teja
CMAP_SEQUENTIAL = "Blues"

plt.rcParams.update({
    "figure.figsize": (8, 5), "figure.dpi": 130, "savefig.dpi": 130,
    "axes.edgecolor": COLOR_BASELINE, "axes.labelcolor": COLOR_INK_PRIMARY,
    "text.color": COLOR_INK_PRIMARY, "xtick.color": COLOR_MUTED,
    "ytick.color": COLOR_MUTED, "grid.color": COLOR_GRID, "axes.grid": True,
    "grid.linewidth": 0.6, "axes.spines.top": False, "axes.spines.right": False,
})
sns.set_style("white")


def guardar_figura(fig, nombre):
    """Guarda en results/figures/{nombre}.png (prefijo sub_05_ o arbol_subrogado_)."""
    os.makedirs("results/figures", exist_ok=True)
    fig.tight_layout()
    fig.savefig(f"results/figures/{nombre}.png", dpi=130, bbox_inches="tight")
    plt.show()


def guardar_tabla(df, nombre, index=False):
    """Guarda en results/tables/{nombre}.csv (prefijo sub_05_)."""
    os.makedirs("results/tables", exist_ok=True)
    ruta = f"results/tables/{nombre}.csv"
    df.to_csv(ruta, index=index)
    return ruta


# --- Carga de los dos bundles y del test ---
bundle1 = joblib.load("results/models/modelo_coste1.joblib")
bundle10 = joblib.load("results/models/modelo_coste10.joblib")
test = pd.read_parquet("data/processed/test.parquet")

FEATURES = list(bundle1["features"])          # 12 columnas, orden canonico del contrato
X_test = test[FEATURES].copy()
y_real = test[TARGET].to_numpy(dtype=int)     # SOLO para contexto; el subrogado NO lo usa

# El estimador supervisado (xgboost) vive en bundle["supervisado"]["modelo"] en 03; en 04 el
# bundle guarda el mismo estimador en bundle["modelo"] (reutilizado, con otro umbral).
modelo_bb = bundle1["supervisado"]["modelo"]  # la CAJA NEGRA (un solo scoring, D-0.2)
umbral_c1 = float(bundle1["umbral"])          # ~0.4934 (simetrico)
umbral_c10 = float(bundle10["umbral"])        # ~0.0906 (asimetrico)

# --- Verificacion explicita de D-0.2: UN scoring, DOS umbrales ---
proba_bb = np.asarray(modelo_bb.predict_proba(X_test))[:, 1].astype("float64")
proba_bb_04 = np.asarray(bundle10["modelo"].predict_proba(X_test))[:, 1].astype("float64")
dif_scoring = float(np.max(np.abs(proba_bb - proba_bb_04)))
assert dif_scoring < 1e-9, f"D-0.2 ROTA: los scorings de 03 y 04 difieren (max|dif|={dif_scoring})"
print(f"[D-0.2] mismo scoring en 03 y 04: max|dif| = {dif_scoring:.2e}  -> confirmado")

# --- LAS ETIQUETAS OBJETIVO DEL SUBROGADO (el corazon del notebook) ---
# NO es y_real. Es lo que la CAJA NEGRA decide. El subrogado imita esto.
y_bb_c1 = cu.aplicar_umbral(proba_bb, umbral_c1)     # decisiones esc1 (simetrico)
y_bb_c10 = cu.aplicar_umbral(proba_bb, umbral_c10)   # decisiones esc2 (asimetrico)

deny_c1 = float(y_bb_c1.mean())
deny_c10 = float(y_bb_c10.mean())

print()
print(f"test: {X_test.shape}  features: {len(FEATURES)}")
print(f"umbral coste1  = {umbral_c1:.6f}  ->  la caja negra deniega al {deny_c1 * 100:.2f}% "
      f"({int(y_bb_c1.sum())} de {len(y_bb_c1)})")
print(f"umbral coste10 = {umbral_c10:.6f}  ->  la caja negra deniega al {deny_c10 * 100:.2f}% "
      f"({int(y_bb_c10.sum())} de {len(y_bb_c10)})")
print()
print(f"[AVISO] prevalencia de 'denegar' en esc1 = {deny_c1 * 100:.2f}%. Un subrogado degenerado")
print(f"        que dijera 'conceder siempre' tendria ya {(1 - deny_c1) * 100:.2f}% de fidelidad")
print(f"        GLOBAL sin aprender nada. Por eso la seccion 5 mide tambien fidelidad por clase.")
print()
print(f"contexto (NO se usa para entrenar): impago real en test = {y_real.mean() * 100:.2f}%")

[D-0.2] mismo scoring en 03 y 04: max|dif| = 0.00e+00  -> confirmado

test: (21000, 12)  features: 12
umbral coste1  = 0.493394  ->  la caja negra deniega al 2.38% (499 de 21000)
umbral coste10 = 0.090609  ->  la caja negra deniega al 18.70% (3926 de 21000)

[AVISO] prevalencia de 'denegar' en esc1 = 2.38%. Un subrogado degenerado
        que dijera 'conceder siempre' tendria ya 97.62% de fidelidad
        GLOBAL sin aprender nada. Por eso la seccion 5 mide tambien fidelidad por clase.

contexto (NO se usa para entrenar): impago real en test = 6.69%


**Hallazgo 1.1 (integridad del artefacto heredado):** Antes de auditar un modelo hay que verificar
que el modelo que se carga ES el modelo que se entreno. Al abrir `modelo_coste1.joblib` con una
version de XGBoost distinta a la del entrenamiento (2.1.4 frente a la 3.2.0 fijada en
`requirements.txt`), el `base_score` aprendido durante el ajuste no se restaura y se reconstruye con
el valor por defecto (0.5). El efecto es silencioso y traicionero: el RANKING se conserva intacto
(AUC 0.870, identico), pero la ESCALA de las probabilidades se infla ~x5 (media 0.33 frente a la
prevalencia real 0.067). Aplicando los umbrales de `03`/`04` sobre ese scoring inflado, el modelo
denegaria al **25%** y al **85%** de los solicitantes en vez de al 2.4% y al 18.7%, y el coste en
test se disparaba de 0.063 a 0.214. La prueba de que se trataba de serializacion y no del modelo:
el `oof_proba` guardado en el MISMO bundle si estaba bien calibrado (media 0.0667 ~ prevalencia
0.0668).

Fijada la version de XGBoost a la del `requirements.txt` (3.2.0), el bundle carga correctamente: la
caja negra deniega al **2.38%** en el escenario simetrico y al **18.70%** en el asimetrico, cifras
que reproducen exactamente las publicadas en `03` y `04`. El modelo auditado en este notebook es,
por tanto, el MISMO que genero los entregables `cs_produccion1.csv` y `cs_produccion2.csv`.
**Leccion de reproducibilidad:** un `joblib.dump` del wrapper de XGBoost NO es portable entre
versiones. La alternativa robusta es serializar el booster en su formato JSON nativo
(`booster.save_model()`), estable entre versiones, y fijar las versiones en `requirements.txt` (ya
hecho).

## 2. Ajuste del arbol subrogado para el escenario coste1

Generar las predicciones de `modelo_coste1` sobre el conjunto de test
(`y_predicha_caja_negra`) y entrenar un `DecisionTreeClassifier` que las imite, siguiendo el
patron descrito en `docs/teoria/subrogados.md` (seccion 3.5).

In [ ]:
# TODO: 2. Ajuste del arbol subrogado para el escenario coste1

## 3. Ajuste del arbol subrogado para el escenario coste10

Analogo a la seccion 2, pero usando `modelo_coste10` y entrenando un arbol subrogado propio e
independiente para este escenario de coste.

In [ ]:
# TODO: 3. Ajuste del arbol subrogado para el escenario coste10

## 4. Extraccion de reglas legibles (ver utilidades referenciadas en docs/teoria/subrogados.md)

Extraer reglas if/then de cada arbol subrogado (soporte y pureza por hoja), guardando el
resultado en `results/tables/reglas_subrogado_coste1.md` y
`results/tables/reglas_subrogado_coste10.md`, junto con las figuras de los arboles en
`results/figures/arbol_subrogado_coste1.png` y `results/figures/arbol_subrogado_coste10.png`.

In [ ]:
# TODO: 4. Extraccion de reglas legibles (ver utilidades referenciadas en docs/teoria/subrogados.md)

## 5. Medicion de fidelidad del subrogado frente al modelo original

Calcular la fidelidad de cada arbol subrogado frente a las predicciones del modelo caja negra
correspondiente (no frente al target real), consolidando los resultados en una tabla
comparativa entre los dos escenarios de coste.

In [ ]:
# TODO: 5. Medicion de fidelidad del subrogado frente al modelo original

## 6. Interpretacion de las reglas obtenidas

Lectura cualitativa de las reglas extraidas en la seccion 4, contrastando que variables y
umbrales aparecen en cada escenario de coste y si la fidelidad es homogenea o varia entre
subgrupos de casos.

In [ ]:
# TODO: 6. Interpretacion de las reglas obtenidas

## 7. Conclusiones

Resumen de resultados de la auditoria con modelo subrogado (fidelidad alcanzada por escenario,
reglas mas relevantes) de cara a `99_ENTREGA.ipynb`.

In [ ]:
# TODO: 7. Conclusiones